In [1]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.hadoop:hadoop-aws:3.3.2,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.180 pyspark-shell"
)


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType
from pyspark.sql.functions import explode, col

linha_schema = StructType([
    StructField("idlinha", StringType(), True),
    StructField("sentido", IntegerType(), True),
    StructField("lat", DoubleType(), True),
    StructField("lng", DoubleType(), True),
    StructField("velocidade", IntegerType(), True),
    StructField("hora", StringType(), True),
])

schema = StructType([
    StructField("hr", StringType(), True),
    StructField("l", ArrayType(linha_schema), True)
])

In [3]:
spark = SparkSession.builder \
    .appName("SparkMinIOExample") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", "yuriadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "yuriadmin") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()

In [4]:
raw_path = "s3a://raw/sptrans/posicoes"  # ajuste para o seu path

df_stream = spark.readStream \
    .schema(schema) \
    .format("json") \
    .option("maxFilesPerTrigger", 1) \
    .load(raw_path)

In [5]:
df_exploded = df_stream.select(
    col("hr"),
    explode(col("l")).alias("onibus")
).select(
    col("hr"),
    col("onibus.idlinha").alias("idlinha"),
    col("onibus.sentido").alias("sentido"),
    col("onibus.lat").alias("lat"),
    col("onibus.lng").alias("lng"),
    col("onibus.velocidade").alias("velocidade"),
    col("onibus.hora").alias("hora_onibus")
)

In [ ]:
query = df_exploded.writeStream \
    .format("parquet") \
    .option("checkpointLocation", "s3a://silver/checkpoints/posicoes_silver") \
    .outputMode("append") \
    .start("s3a://silver/posicoes")

query.awaitTermination()